# NLP Gensim Tutorial Notebook

This notebook introduces core **Gensim** workflows in a cleaner, more classroom-friendly format.

The original article covers:
- corpus creation
- Bag of Words and TF-IDF
- bigrams and trigrams
- Word2Vec and Doc2Vec
- topic modelling
- similarity computation

This notebook follows the same structure, but uses compact examples that are easier to run in a notebook.

## 1. Optional Colab Setup

If you run this notebook in Google Colab, the next cell installs the packages used here.

In [ ]:
import sys
import subprocess

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gensim', 'nltk', 'matplotlib', 'pandas'])
    print('Installed packages for Colab.')
else:
    print('Not running in Google Colab. Skipping installation.')


## 2. Imports

In [2]:
!pip install nltk

   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 1.6/1.6 MB 6.9 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gensim
import gensim.downloader as api
from gensim import corpora, models
from gensim.utils import simple_preprocess
from gensim.models import Word2Vec, Doc2Vec, LdaModel, LsiModel
from gensim.models.doc2vec import TaggedDocument
from gensim.models.phrases import Phrases, Phraser
from gensim.similarities import SparseTermSimilarityMatrix, WordEmbeddingSimilarityIndex, SoftCosineSimilarity

from nltk.corpus import stopwords
import nltk

nltk.download('stopwords')

plt.rcParams['figure.figsize'] = (8, 5)
pd.set_option('display.max_colwidth', 200)

print('gensim version:', gensim.__version__)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...


gensim version: 4.4.0


[nltk_data]   Unzipping corpora\stopwords.zip.


In [4]:
# Explore datasets available through gensim.downloader
dataset_info = api.info()['corpora']

datasets_df = pd.DataFrame(dataset_info).T
display(datasets_df[['num_records', 'description']].head(15))

print('Total available corpora:', len(datasets_df))
print('Example dataset names:')
print(list(datasets_df.index[:10]))


,num_records,description
semeval-2016-2017-task3-subtaskBC,-1,"SemEval 2016 / 2017 Task 3 Subtask B and C datasets contain train+development (317 original questions, 3,169 related questions, and 31,690 comments), and test datasets in English. The description ..."
semeval-2016-2017-task3-subtaskA-unannotated,189941,"SemEval 2016 / 2017 Task 3 Subtask A unannotated dataset contains 189,941 questions and 1,894,456 comments in English collected from the Community Question Answering (CQA) web forum of Qatar Livin..."
patent-2017,353197,"Patent Grant Full Text. Contains the full text including tables, sequence data and 'in-line' mathematical expressions of each patent grant issued in 2017."
quora-duplicate-questions,404290,"Over 400,000 lines of potential question duplicate pairs. Each line contains IDs for each question in the pair, the full text for each question, and a binary value that indicates whether the line ..."
wiki-english-20171001,4924894,Extracted Wikipedia dump from October 2017. Produced by `python -m gensim.scripts.segment_wiki -f enwiki-20171001-pages-articles.xml.bz2 -o wiki-en.gz`
text8,1701,"First 100,000,000 bytes of plain text from Wikipedia. Used for testing purposes; see wiki-english-* for proper full Wikipedia datasets."
fake-news,12999,"News dataset, contains text and metadata from 244 websites and represents 12,999 posts in total from a specific window of 30 days. The data was pulled using the webhose.io API, and because it's co..."
20-newsgroups,18846,"The notorious collection of approximately 20,000 newsgroup posts, partitioned (nearly) evenly across 20 different newsgroups."
__testing_matrix-synopsis,NaN,[THIS IS ONLY FOR TESTING] Synopsis of the movie matrix.
__testing_multipart-matrix-synopsis,NaN,[THIS IS ONLY FOR TESTING] Synopsis of the movie matrix.


Total available corpora: 10
Example dataset names:
['semeval-2016-2017-task3-subtaskBC', 'semeval-2016-2017-task3-subtaskA-unannotated', 'patent-2017', 'quora-duplicate-questions', 'wiki-english-20171001', 'text8', 'fake-news', '20-newsgroups', '__testing_matrix-synopsis', '__testing_multipart-matrix-synopsis']


## 3. Common Terminologies

Some key terms from the tutorial:

- **Corpus**: a collection of documents
- **Token**: an individual word/subword unit after preprocessing
- **Dictionary**: a mapping from tokens to integer ids
- **Bag of Words (BoW)**: sparse representation of token counts per document
- **TF-IDF**: weighting scheme that down-weights common words
- **Topic model**: model that discovers hidden themes across documents
- **Embedding**: dense vector representation for words or documents


## 4. Create a Small Example Corpus

The article uses either a text file or Gensim downloader datasets. For a compact notebook example, we will start with a small custom dataset.

In [6]:
documents = [
    'Natural Language Processing (NLP) helps computers understand HUMAN language!!!',
    'Gensim is a useful Python library for topic-modelling, similarity, and word embeddings in 2025.',
    'Word2Vec learns vector representations of words from text corpora... even when text is messy.',
    'Doc2Vec extends Word2Vec to represent WHOLE documents as vectors, not just words.',
    'Topic models such as LDA help discover hidden themes in document collections; they are unsupervised.',
    'TF-IDF helps identify words that are important inside a document corpus, rather than just frequent.',
    'Bigrams and trigrams capture phrases such as machine learning, deep learning, and neural networks.',
    'Similarity models can compare documents using semantic vector representations across domains.',
    'Caf\u00e9, na\u00efve, and r\u00e9sum\u00e9 are useful examples for testing accent normalization.',
    'Email-like text such as nlp_course@example.com or tokens like GPT-4-style should be cleaned carefully.'
]

pd.DataFrame({'document': documents})


,document
0,Natural language processing helps computers understand human language.
1,Gensim is a useful Python library for topic modelling and word embeddings.
2,Word2Vec learns vector representations of words from text corpora.
3,Doc2Vec extends Word2Vec to represent whole documents as vectors.
4,Topic models such as LDA help discover hidden themes in document collections.
5,TF IDF helps identify words that are important inside a document corpus.
6,Bigrams and trigrams capture phrases such as machine learning and deep learning.
7,Similarity models can compare documents using semantic vector representations.


## 5. Preprocess the Dataset

We use `simple_preprocess()` to turn raw text into tokens.

In practice, `simple_preprocess()` does a few useful things for us:

- converts text to lowercase
- splits text into word-like tokens
- removes punctuation by keeping alphabetic tokens
- drops very short or very long tokens by default
- returns a clean Python list of tokens

For example, a sentence like `"Gensim is GREAT for NLP!"` becomes something close to `['gensim', 'is', 'great', 'for', 'nlp']`.

We also set `deacc=True`, which removes accent marks and strips punctuation-like characters more aggressively.

In [7]:
tokenized_docs = [simple_preprocess(doc, deacc=True) for doc in documents]
tokenized_docs


[['natural',
  'language',
  'processing',
  'helps',
  'computers',
  'understand',
  'human',
  'language'],
 ['gensim',
  'is',
  'useful',
  'python',
  'library',
  'for',
  'topic',
  'modelling',
  'and',
  'word',
  'embeddings'],
 ['word',
  'vec',
  'learns',
  'vector',
  'representations',
  'of',
  'words',
  'from',
  'text',
  'corpora'],
 ['doc',
  'vec',
  'extends',
  'word',
  'vec',
  'to',
  'represent',
  'whole',
  'documents',
  'as',
  'vectors'],
 ['topic',
  'models',
  'such',
  'as',
  'lda',
  'help',
  'discover',
  'hidden',
  'themes',
  'in',
  'document',
  'collections'],
 ['tf',
  'idf',
  'helps',
  'identify',
  'words',
  'that',
  'are',
  'important',
  'inside',
  'document',
  'corpus'],
 ['bigrams',
  'and',
  'trigrams',
  'capture',
  'phrases',
  'such',
  'as',
  'machine',
  'learning',
  'and',
  'deep',
  'learning'],
 ['similarity',
  'models',
  'can',
  'compare',
  'documents',
  'using',
  'semantic',
  'vector',
  'representatio

In [8]:
# Similar preprocessing without gensim, using lower-level Python steps
def manual_preprocess(text, min_len=2, max_len=15):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = text.split()
    tokens = [token for token in tokens if min_len <= len(token) <= max_len]
    return tokens

manual_tokenized_docs = [manual_preprocess(doc) for doc in documents]

comparison_df = pd.DataFrame({
    'original': documents,
    'gensim_simple_preprocess': tokenized_docs,
    'manual_preprocess': manual_tokenized_docs,
})

comparison_df


,original,gensim_simple_preprocess,manual_preprocess
0,Natural language processing helps computers understand human language.,"[natural, language, processing, helps, computers, understand, human, language]","[natural, language, processing, helps, computers, understand, human, language]"
1,Gensim is a useful Python library for topic modelling and word embeddings.,"[gensim, is, useful, python, library, for, topic, modelling, and, word, embeddings]","[gensim, is, useful, python, library, for, topic, modelling, and, word, embeddings]"
2,Word2Vec learns vector representations of words from text corpora.,"[word, vec, learns, vector, representations, of, words, from, text, corpora]","[word, vec, learns, vector, representations, of, words, from, text, corpora]"
3,Doc2Vec extends Word2Vec to represent whole documents as vectors.,"[doc, vec, extends, word, vec, to, represent, whole, documents, as, vectors]","[doc, vec, extends, word, vec, to, represent, whole, documents, as, vectors]"
4,Topic models such as LDA help discover hidden themes in document collections.,"[topic, models, such, as, lda, help, discover, hidden, themes, in, document, collections]","[topic, models, such, as, lda, help, discover, hidden, themes, in, document, collections]"
5,TF IDF helps identify words that are important inside a document corpus.,"[tf, idf, helps, identify, words, that, are, important, inside, document, corpus]","[tf, idf, helps, identify, words, that, are, important, inside, document, corpus]"
6,Bigrams and trigrams capture phrases such as machine learning and deep learning.,"[bigrams, and, trigrams, capture, phrases, such, as, machine, learning, and, deep, learning]","[bigrams, and, trigrams, capture, phrases, such, as, machine, learning, and, deep, learning]"
7,Similarity models can compare documents using semantic vector representations.,"[similarity, models, can, compare, documents, using, semantic, vector, representations]","[similarity, models, can, compare, documents, using, semantic, vector, representations]"


## 6. Create a Dictionary and a Bag-of-Words Corpus

In [ ]:
dictionary = corpora.Dictionary(tokenized_docs)
bow_corpus = [dictionary.doc2bow(doc) for doc in tokenized_docs]

print('Dictionary size:', len(dictionary))
print('Token to id mapping sample:', list(dictionary.token2id.items())[:15])
print('\nFirst BoW document:')
print(bow_corpus[0])


In [ ]:
def bow_to_words(bow_doc, dictionary):
    return [(dictionary[token_id], count) for token_id, count in bow_doc]

bow_to_words(bow_corpus[0], dictionary)


## 7. Create a TF-IDF Representation

This follows the tutorial section that converts BoW counts into TF-IDF weights.

In [ ]:
tfidf_model = models.TfidfModel(bow_corpus)
tfidf_corpus = list(tfidf_model[bow_corpus])

def tfidf_to_words(tfidf_doc, dictionary):
    return [(dictionary[token_id], round(weight, 3)) for token_id, weight in tfidf_doc]

tfidf_to_words(tfidf_corpus[0], dictionary)


## 8. Create Bigrams and Trigrams with Gensim

We build bigrams and trigrams with `Phrases` below.

In [ ]:
bigram = Phrases(tokenized_docs, min_count=1, threshold=1)
bigram_phraser = Phraser(bigram)

bigram_docs = [bigram_phraser[doc] for doc in tokenized_docs]
bigram_docs


In [ ]:
trigram = Phrases(bigram_docs, min_count=1, threshold=1)
trigram_phraser = Phraser(trigram)

trigram_docs = [trigram_phraser[doc] for doc in bigram_docs]
trigram_docs


## 9. Train a Word2Vec Model

A common Word2Vec demo uses the `text8` dataset. To keep this notebook lightweight, we train on our own small tokenized corpus.

In [ ]:
w2v_model = Word2Vec(
    sentences=trigram_docs,
    vector_size=50,
    window=3,
    min_count=1,
    workers=1,
    epochs=100,
    seed=42,
)

print('Vocabulary size:', len(w2v_model.wv))
print('Vector for "gensim":')
w2v_model.wv['gensim'][:10]


In [ ]:
w2v_model.wv.most_similar('document', topn=5)


## 10. Train a Doc2Vec Model

Doc2Vec gives a vector representation for a whole document, not just a word.

In [ ]:
tagged_docs = [TaggedDocument(words=doc, tags=[f'DOC_{i}']) for i, doc in enumerate(trigram_docs)]
tagged_docs[:2]


In [ ]:
d2v_model = Doc2Vec(vector_size=40, min_count=1, epochs=80, seed=42)
d2v_model.build_vocab(tagged_docs)
d2v_model.train(tagged_docs, total_examples=d2v_model.corpus_count, epochs=d2v_model.epochs)

print('Doc2Vec training complete.')
print('Vector sample for DOC_0:')
d2v_model.dv['DOC_0'][:10]


In [ ]:
new_doc = simple_preprocess('semantic document similarity with gensim vectors', deacc=True)
inferred_vector = d2v_model.infer_vector(new_doc)
similar_docs = d2v_model.dv.most_similar([inferred_vector], topn=3)

print('New tokens:', new_doc)
print('Most similar documents:', similar_docs)


## 11. Topic Modelling with LDA

Here we apply LDA after preprocessing and building a dictionary/corpus.

In [ ]:
lda_model = LdaModel(
    corpus=bow_corpus,
    id2word=dictionary,
    num_topics=3,
    random_state=42,
    passes=20,
)

lda_topics = lda_model.print_topics(num_words=6)
lda_topics


In [ ]:
for i, bow_doc in enumerate(bow_corpus):
    print(f'Document {i}:', lda_model.get_document_topics(bow_doc))


## 12. Topic Modelling with LSI

LSI is another classic topic-model style approach.

In [ ]:
lsi_model = LsiModel(corpus=bow_corpus, id2word=dictionary, num_topics=2)
lsi_model.print_topics(num_topics=2)


## 13. Compute Similarity with Soft Cosine Similarity

Here we compute soft cosine similarity with word embeddings. Instead of a huge pre-trained embedding model, we reuse the Word2Vec model we trained above.

In [ ]:
sim_docs = [
    simple_preprocess('Afghanistan is an Asian country and Kabul is its capital.'),
    simple_preprocess('India is an Asian country and Delhi is its capital.'),
    simple_preprocess('Greece is a European country and Athens is its capital.'),
]

sim_dictionary = corpora.Dictionary(sim_docs)
sim_corpus = [sim_dictionary.doc2bow(doc) for doc in sim_docs]

sim_docs


In [ ]:
similarity_w2v = Word2Vec(
    sentences=sim_docs + trigram_docs,
    vector_size=50,
    min_count=1,
    workers=1,
    epochs=100,
    seed=42,
)

similarity_index = WordEmbeddingSimilarityIndex(similarity_w2v.wv)
similarity_matrix = SparseTermSimilarityMatrix(similarity_index, sim_dictionary)
soft_cosine = SoftCosineSimilarity(sim_corpus, similarity_matrix, num_best=3)

query = sim_dictionary.doc2bow(sim_docs[0])
soft_cosine[query]


## 14. More Word2Vec Similarity Operations

This section shows a few useful vector-space queries like nearest neighbors and similarity comparisons.

In [ ]:
print('Similarity(document, corpus):', round(w2v_model.wv.similarity('document', 'corpus'), 4))
print('Similarity(gensim, embeddings):', round(w2v_model.wv.similarity('gensim', 'embeddings'), 4))
print('\nMost similar to "topic":')
print(w2v_model.wv.most_similar('topic', topn=5))


## 15. Important Practical Note

Older Gensim tutorials often included sections on **text summarization** and **keyword extraction** using `gensim.summarization`.

That functionality existed in older Gensim 3.x documentation, but modern Gensim 4.x installations often do not include those APIs directly.
So this notebook focuses on the parts that are still the most reliable in current Gensim workflows:

- corpus building
- TF-IDF
- phrases
- Word2Vec
- Doc2Vec
- LDA / LSI
- semantic similarity


## 16. Summary

In this notebook, we built a runnable walkthrough of core Gensim workflows.

You saw how to:
- preprocess text with `simple_preprocess`
- build dictionaries and Bag-of-Words corpora
- compute TF-IDF weights
- detect bigrams and trigrams
- train Word2Vec and Doc2Vec models
- build LDA and LSI topic models
- compute semantic similarity using Soft Cosine Similarity

This gives a solid practical overview of what Gensim is designed for.